# CS 5330 — Week 6 Bonus Activity: Can CLIP Read?

In the workbook you saw how a Vision Transformer turns an image into a sequence of patch tokens. Today you will use one: the image encoder inside **CLIP** is a ViT-B/32, the 32 × 32-patch model from question 4f.

The plan has two rounds:

1. **Round 1:** ask CLIP to name the fruit in a set of ordinary photos.
2. **Round 2:** ask again, after a sticky note with the name of a *different* fruit has been stuck on each one.

Then you will try to understand what happened, and try to fix it.

**Time:** about 25 minutes. **Runtime:** a GPU helps but is not required; everything here runs on a CPU in under a minute.

*Note on spelling: the text in this notebook uses Canadian spelling ("colour"), while the code uses US spelling, as most libraries do.*

## How CLIP works (2-minute read)

CLIP (Radford et al., 2021) has two encoders: a **text encoder** (a transformer that reads captions) and an **image encoder** (a ViT). OpenAI trained them together on about 400 million image–caption pairs from the web. The training goal was simple: the embedding of an image should be close to the embedding of its own caption and far from the embeddings of other captions.

That makes **zero-shot classification** possible. To classify an image without any training:

1. Write one caption per class, for example *"a photo of a banana"*.
2. Embed the image and every caption.
3. Pick the caption whose embedding is most similar (by cosine similarity) to the image embedding. A softmax over the similarities turns them into probabilities.

You never fine-tune anything. Keep in mind what CLIP learned from, though: web images *and the text around them*. That detail will matter in Round 2.

## 1. Setup

In [ ]:
import re
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image, ImageDraw, ImageOps
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "openai/clip-vit-base-patch32"  # ViT-B/32 image encoder

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME} on {DEVICE}")

## 2. Load the photos

The photos follow a naming convention, so the notebook can read the true fruit and the word on the note straight from the filename:

| Filename | Meaning |
|---|---|
| `apple_plain.jpg` | an apple, no note |
| `apple_note-banana.jpg` | the same apple with a sticky note that says "banana" |

You do not need to change anything in the next cell unless your instructor tells you to.

In [ ]:
# --- Configuration -------------------------------------------------------
# PHOTO_SOURCE: "github" downloads the class photos; "upload" lets you pick files.
PHOTO_SOURCE = "github"
GITHUB_BASE_URL = "https://raw.githubusercontent.com/LinoNU/cs5330-produce-photos/main/week6_clip/"
FILENAMES = [
    "apple_plain.jpg", "apple_note-banana.jpg",
    "pear_plain.jpg", "pear_note-banana.jpg",
    "tomato_plain.jpg", "tomato_note-banana.jpg",
]
# -------------------------------------------------------------------------

FILENAME_PATTERN = re.compile(r"^(?P<fruit>[a-z]+)_(?:plain|note-(?P<written>[a-z]+))\.(?:jpe?g|png)$")


@dataclass
class Photo:
    """One test photo and the metadata encoded in its filename."""
    name: str
    image: Image.Image
    fruit: str                 # what the fruit really is
    written: str | None        # word on the sticky note, or None if there is no note


def parse_filename(name: str) -> tuple[str, str | None]:
    """Return (true fruit, written word or None) from a filename like 'banana_note-apple.jpg'."""
    match = FILENAME_PATTERN.match(name.lower())
    if match is None:
        raise ValueError(f"Filename '{name}' does not follow '<fruit>_plain.jpg' or '<fruit>_note-<word>.jpg'")
    return match["fruit"], match["written"]


def open_image(data: bytes) -> Image.Image:
    """Decode image bytes, apply the phone's rotation flag, and convert to RGB."""
    return ImageOps.exif_transpose(Image.open(BytesIO(data))).convert("RGB")


def load_photos() -> list[Photo]:
    """Load every photo from GitHub or from a Colab upload, depending on PHOTO_SOURCE."""
    if PHOTO_SOURCE == "github":
        files = {}
        for name in FILENAMES:
            response = requests.get(GITHUB_BASE_URL + name, timeout=30)
            response.raise_for_status()
            files[name] = response.content
    elif PHOTO_SOURCE == "upload":
        from google.colab import files as colab_files
        files = colab_files.upload()
    else:
        raise ValueError(f"Unknown PHOTO_SOURCE: {PHOTO_SOURCE}")

    photos = []
    for name, data in sorted(files.items()):
        fruit, written = parse_filename(Path(name).name)
        photos.append(Photo(name=name, image=open_image(data), fruit=fruit, written=written))
    return photos


photos = load_photos()
plain_photos = [p for p in photos if p.written is None]
note_photos = [p for p in photos if p.written is not None]
print(f"Loaded {len(plain_photos)} plain photos and {len(note_photos)} sticky-note photos")

## 3. A zero-shot classifier in a few lines

The function below follows the three steps from the reading. The candidate labels include every fruit in the photos, every word written on a note (so CLIP always has the "trick" answer available), and a few extra fruits that do not appear in any photo. The extras make the task a little harder than a three- or four-way choice.

In [ ]:
DEFAULT_TEMPLATE = "a photo of a {}"
DISTRACTOR_LABELS = ["orange", "lemon", "peach", "strawberry"]  # fruits that are not in any photo


def candidate_labels(photo_set: list[Photo], extras: list[str] = DISTRACTOR_LABELS) -> list[str]:
    """True fruits, written words and distractors, without duplicates, in a stable order."""
    labels = {p.fruit for p in photo_set} | {p.written for p in photo_set if p.written} | set(extras)
    return sorted(labels)


@torch.inference_mode()
def classify(image: Image.Image, labels: list[str], template: str = DEFAULT_TEMPLATE) -> dict[str, float]:
    """Return a {label: probability} dict for one image, sorted from most to least likely."""
    prompts = [template.format(label) for label in labels]
    inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True).to(DEVICE)
    # logits_per_image = scaled cosine similarity between the image and each prompt
    probs = model(**inputs).logits_per_image.softmax(dim=-1).squeeze(0).cpu()
    ranked = sorted(zip(labels, probs.tolist()), key=lambda pair: pair[1], reverse=True)
    return dict(ranked)


def show_predictions(photo: Photo, probs: dict[str, float], top_k: int = 5, title: str | None = None) -> None:
    """Show the photo next to a bar chart of CLIP's top-k labels (true fruit in black, top guess in red)."""
    top = list(probs.items())[:top_k]
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 1.4]})
    ax_img.imshow(photo.image)
    ax_img.axis("off")
    ax_img.set_title(title or photo.name, fontsize=10)

    names = [name for name, _ in top][::-1]
    values = [value for _, value in top][::-1]
    colours = ["#C8102E" if name == top[0][0] else "#BFBFBF" for name in names]
    ax_bar.barh(names, values, color=colours)
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("probability")
    for tick in ax_bar.get_yticklabels():
        if tick.get_text() == photo.fruit:
            tick.set_fontweight("bold")
    ax_bar.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


LABELS = candidate_labels(photos)
print("Candidate labels:", LABELS)

## 4. Round 1: ordinary fruit

Run the cell below. The true fruit is shown in **bold** on each chart, and CLIP's top guess is the red bar.

In [ ]:
round1_correct = 0
for photo in plain_photos:
    probs = classify(photo.image, LABELS)
    show_predictions(photo, probs)
    round1_correct += next(iter(probs)) == photo.fruit

print(f"Round 1 accuracy: {round1_correct}/{len(plain_photos)}")

### Your turn

**Q1.** How confident was CLIP in Round 1? Did any photo fool it, and if so, can you think of a reason?

*Your answer:*

**Q2. Predict before you run Round 2.** Each fruit now has a sticky note that says "banana." Write down what you think CLIP will predict for each photo, and roughly how confident it will be.

*Your prediction:*

## 5. Round 2: the same fruit, plus a sticky note

In [ ]:
round2_correct = round2_fooled = 0
for photo in note_photos:
    probs = classify(photo.image, LABELS)
    show_predictions(photo, probs, title=f"{photo.name}  (note says '{photo.written}')")
    top_guess = next(iter(probs))
    round2_correct += top_guess == photo.fruit
    round2_fooled += top_guess == photo.written

print(f"Round 2: {round2_correct}/{len(note_photos)} correct, "
      f"{round2_fooled}/{len(note_photos)} predicted the word on the note")

### Your turn

**Q3.** How did Round 2 compare with your prediction? Which result surprised you most?

*Your answer:*

**Q4.** Why would a model trained on web images and captions learn to *read*? Think about what kinds of images on the web contain text, and what their captions tend to say.

*Your answer:*

This is called a **typographic attack**. OpenAI researchers reported it shortly after CLIP's release (Goh et al., 2021): an apple with a handwritten label saying "iPod" was classified as an iPod. Next week we will look at adversarial examples more generally, including attacks you cannot see with your eyes.

## 6. Can we fix it?

Try one or more of the experiments below. Each one only takes a small edit.

**Experiment A: change the prompt.** Does a different caption template help? Try ideas like `"a photo of a {}, a type of fruit"` or `"a close-up photo of a {} on a table"`.

In [ ]:
NEW_TEMPLATE = "a photo of a {}, a type of fruit"  # edit me

for photo in note_photos:
    show_predictions(photo, classify(photo.image, LABELS, template=NEW_TEMPLATE),
                     title=f"{photo.name}  |  template: {NEW_TEMPLATE!r}")

**Experiment B: give CLIP an "honest" option.** Add labels that describe what is really in the photo, such as `"apple with a sticky note on it"`. Does CLIP prefer the honest description over the trick label?

In [ ]:
EXTRA_LABELS = [f"{p.fruit} with a sticky note on it" for p in note_photos]  # edit or add your own

for photo in note_photos:
    show_predictions(photo, classify(photo.image, LABELS + EXTRA_LABELS))

**Experiment C: cover the note.** Paint a grey box over the note and classify again. The box is given as fractions of the image width and height, `(left, top, right, bottom)`, so `(0.3, 0.2, 0.7, 0.5)` covers the middle-top region. Adjust it for one photo until the note is hidden.

In [ ]:
def cover_region(image: Image.Image, box: tuple[float, float, float, float], fill: str = "#808080") -> Image.Image:
    """Return a copy of the image with a solid rectangle over box = (left, top, right, bottom) in [0, 1] fractions."""
    left, top, right, bottom = box
    if not (0 <= left < right <= 1 and 0 <= top < bottom <= 1):
        raise ValueError("box must satisfy 0 <= left < right <= 1 and 0 <= top < bottom <= 1")
    covered = image.copy()
    width, height = covered.size
    ImageDraw.Draw(covered).rectangle(
        [left * width, top * height, right * width, bottom * height], fill=fill)
    return covered


PHOTO_INDEX = 0                  # which sticky-note photo to edit
BOX = (0.30, 0.20, 0.70, 0.50)   # edit me

target = note_photos[PHOTO_INDEX]
covered = Photo(name=target.name, image=cover_region(target.image, BOX), fruit=target.fruit, written=target.written)
show_predictions(covered, classify(covered.image, LABELS), title=f"{target.name}  (note covered)")

### Your turn

**Q5.** Which experiment worked best? Would that fix be practical in a real system, where you cannot hand-pick the prompt or the box for every image?

*Your answer:*

**Q6. Back to produce inspection.** Real fruit at the grocery store often carries PLU stickers with printed numbers and sometimes words. If you were deploying CLIP to sort produce on a conveyor belt, what risks would you test for before trusting it?

*Your answer:*

## Optional: bring your own trick

Write a word on a piece of paper, put it on any object near you, and take a photo with your phone. Set `PHOTO_SOURCE = "upload"` in Section 2, name the file following the convention (for example `mug_note-banana.jpg`), and run the notebook again. Can you find a word that fools CLIP even on an object that looks nothing like it?

---

### References

- Goh, G. et al. (2021). Multimodal neurons in artificial neural networks. *Distill*.
- Radford, A. et al. (2021). Learning transferable visual models from natural language supervision. *ICML*.